In [5]:
import pandas as pd

df = pd.read_csv("../data/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv")
df.columns = df.columns.str.strip() # CIC-IDS2017 headers have stray spaces
print(df.shape)
print(df["Label"].value_counts())

(286467, 85)
Label
PortScan    158930
BENIGN      127537
Name: count, dtype: int64


In [8]:
print("Unique source IPs:", df["Source IP"].nunique())
print("Unique destination IPs:", df["Destination IP"].nunique())

attack = df[df["Label"] == "PortScan"]
print("\nSource IPs in attack flows:")
print(attack["Source IP"].value_counts().head(10))

# For each source IP: how many flows, and what fraction are attacks?
by_host = df.groupby("Source IP")["Label"].agg(
    flows="count",
    attack_flows=lambda s: (s == "PortScan").sum()
)
by_host["attack_frac"] = by_host["attack_flows"] / by_host["flows"]
print("\nHosts with any attack flows:", (by_host["attack_flows"] > 0).sum())
print(by_host.sort_values("attack_flows", ascending=False).head(10))

Unique source IPs: 3667
Unique destination IPs: 4401

Source IPs in attack flows:
Source IP
172.16.0.1    158930
Name: count, dtype: int64

Hosts with any attack flows: 1
                 flows  attack_flows  attack_frac
Source IP                                        
172.16.0.1      158932        158930     0.999987
101.37.178.168       1             0     0.000000
52.30.90.179         4             0     0.000000
52.31.213.44         2             0     0.000000
52.32.174.154        2             0     0.000000
52.32.39.49          1             0     0.000000
52.33.209.128        4             0     0.000000
52.33.61.7           1             0     0.000000
52.34.178.251        2             0     0.000000
52.35.101.212        2             0     0.000000


In [9]:
df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
print("Unparsed timestamps:", df["Timestamp"].isna().sum())
print("Time range:", df["Timestamp"].min(), "to", df["Timestamp"].max())

df["window"] = df["Timestamp"].dt.floor("5min")
df["is_attack"] = (df["Label"] == "PortScan").astype(int)
hw = df.groupby(["Source IP", "window"]).agg(
    flows=("is_attack", "size"),
    attack_flows=("is_attack", "sum"),
).reset_index()
hw["label"] = (hw["attack_flows"] > 0).astype(int)

print("\nHost-window rows:", len(hw))
print(hw["label"].value_counts())
print("Positive fraction:", hw["label"].mean())

Unparsed timestamps: 0
Time range: 2017-07-07 01:00:00 to 2017-07-07 03:29:00

Host-window rows: 7965
label
0    7953
1      12
Name: count, dtype: int64
Positive fraction: 0.0015065913370998117


In [10]:
# helper columns at the flow level
df["no_reply"] = (df["Total Backward Packets"] == 0).astype(int)
df["rst"] = (df["RST Flag Count"] > 0).astype(int)

feat = df.groupby(["Source IP", "window"]).agg(
    flows=("is_attack", "size"),
    n_dst_ports=("Destination Port", "nunique"),
    n_dst_ips=("Destination IP", "nunique"),
    mean_fwd_pkts=("Total Fwd Packets", "mean"),
    mean_bwd_pkts=("Total Backward Packets", "mean"),
    mean_duration=("Flow Duration", "mean"),
    no_reply_frac=("no_reply", "mean"),
    rst_frac=("rst", "mean"),
    label=("is_attack", "max"),     # 1 if any attack flow in the window
).reset_index()

feat["ports_per_flow"] = feat["n_dst_ports"] / feat["flows"]

print(feat.shape)
print(feat.groupby("label")[["flows", "n_dst_ports", "no_reply_frac",
                             "rst_frac", "mean_fwd_pkts"]].mean())

# dumb baseline: rank by flow count only
top = feat.sort_values("flows", ascending=False).head(12)
print("\nBaseline precision@12 (rank by flows):", top["label"].mean())

(7965, 12)
              flows  n_dst_ports  no_reply_frac  rst_frac  mean_fwd_pkts
label                                                                   
0         16.036213     2.837671       0.409733  0.000005       2.605812
1      13244.250000   417.916667       0.024658  0.000000       2.047168

Baseline precision@12 (rank by flows): 0.4166666666666667


In [11]:
cols = ["Source IP", "window", "flows", "n_dst_ports", "n_dst_ips", "no_reply_frac", "label"]

print("Top 12 by flows:")
print(feat.sort_values("flows", ascending=False).head(12)[cols].to_string(index=False))

print("\nAll 12 attack windows:")
print(feat[feat["label"] == 1].sort_values("flows", ascending=False)[cols].to_string(index=False))

top_ports = feat.sort_values("n_dst_ports", ascending=False).head(12)
print("\nPrecision@12 ranking by n_dst_ports:", top_ports["label"].mean())

Top 12 by flows:
    Source IP              window  flows  n_dst_ports  n_dst_ips  no_reply_frac  label
   172.16.0.1 2017-07-07 02:50:00  93097          998          2       0.000000      1
   172.16.0.1 2017-07-07 02:55:00  45397          996          1       0.000000      1
   172.16.0.1 2017-07-07 03:05:00   9027          999          1       0.001994      1
   172.16.0.1 2017-07-07 03:20:00   9025          999          1       0.001994      1
192.168.10.14 2017-07-07 01:35:00   2607            9        353       0.199079      0
192.168.10.19 2017-07-07 01:35:00   2081            7        188       0.240750      0
192.168.10.14 2017-07-07 01:30:00   2039            6        292       0.072094      0
   172.16.0.1 2017-07-07 03:10:00   2011         1000          1       0.001989      1
192.168.10.17 2017-07-07 03:25:00   1633            9        104       0.172076      0
192.168.10.14 2017-07-07 01:45:00   1579            8        158       0.103230      0
 192.168.10.8 2017-07-07 0

In [12]:
# attack fraction per window
agg = df.groupby(["Source IP", "window"]).agg(
    attack_flows=("is_attack", "sum"),
    flows_=("is_attack", "size"),
).reset_index()
agg["attack_frac"] = agg["attack_flows"] / agg["flows_"]

feat = feat.merge(agg[["Source IP", "window", "attack_flows", "attack_frac"]],
                  on=["Source IP", "window"])

print(feat[feat["label"] == 1]
      .sort_values("flows", ascending=False)
      [["window", "flows", "attack_flows", "attack_frac", "n_dst_ports"]]
      .to_string(index=False))

# evaluation helpers
def precision_at_k(data, score_col, k):
    return data.sort_values(score_col, ascending=False).head(k)["label"].mean()

def recall_at_k(data, score_col, k):
    top = data.sort_values(score_col, ascending=False).head(k)
    return top["label"].sum() / data["label"].sum()

for col in ["flows", "n_dst_ports"]:
    print(col, "P@5:", precision_at_k(feat, col, 5),
          "R@12:", round(recall_at_k(feat, col, 12), 2))

             window  flows  attack_flows  attack_frac  n_dst_ports
2017-07-07 02:50:00  93097         93096     0.999989          998
2017-07-07 02:55:00  45397         45397     1.000000          996
2017-07-07 03:05:00   9027          9027     1.000000          999
2017-07-07 03:20:00   9025          9025     1.000000          999
2017-07-07 03:10:00   2011          2011     1.000000         1000
2017-07-07 01:55:00    177           177     1.000000            5
2017-07-07 02:10:00     70            70     1.000000            5
2017-07-07 02:15:00     68            68     1.000000            5
2017-07-07 02:00:00     49            49     1.000000            5
2017-07-07 01:05:00      7             7     1.000000            1
2017-07-07 03:00:00      2             2     1.000000            1
2017-07-07 01:50:00      1             1     1.000000            1
flows P@5: 0.8 R@12: 0.42
n_dst_ports P@5: 1.0 R@12: 0.42


In [13]:
feat["flows_per_dst_ip"] = feat["flows"] / feat["n_dst_ips"]
feat["flows_per_dst_port"] = feat["flows"] / feat["n_dst_ports"]

small_attack = feat[(feat["label"] == 1) & (feat["n_dst_ports"] <= 5)]
busy_benign = feat[(feat["label"] == 0) & (feat["flows"] >= 100)]

cols = ["flows", "n_dst_ports", "n_dst_ips", "flows_per_dst_ip",
        "flows_per_dst_port", "mean_fwd_pkts", "mean_bwd_pkts", "mean_duration"]

print("small attack windows:", len(small_attack))
print(small_attack[cols].mean().round(2))
print("\nbusy benign windows:", len(busy_benign))
print(busy_benign[cols].mean().round(2))

small attack windows: 7
flows                      53.43
n_dst_ports                 3.29
n_dst_ips                   1.00
flows_per_dst_ip           53.43
flows_per_dst_port         11.83
mean_fwd_pkts               2.77
mean_bwd_pkts               1.54
mean_duration         1983306.17
dtype: float64

busy benign windows: 222
flows                      458.66
n_dst_ports                 13.89
n_dst_ips                   55.77
flows_per_dst_ip            25.10
flows_per_dst_port          61.70
mean_fwd_pkts                7.72
mean_bwd_pkts                8.27
mean_duration         14219149.24
dtype: float64


In [14]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

feat["dur_s"] = feat["mean_duration"].clip(lower=0) / 1e6
cols = ["flows", "n_dst_ports", "n_dst_ips", "flows_per_dst_ip",
        "flows_per_dst_port", "mean_fwd_pkts", "mean_bwd_pkts",
        "dur_s", "no_reply_frac"]

X = np.log1p(feat[cols].clip(lower=0))   # log tames huge ranges
y = feat["label"]
cut = pd.Timestamp("2017-07-07 02:30:00")

def run(train_mask, name):
    test_mask = ~train_mask
    scaler = StandardScaler().fit(X[train_mask])
    model = LogisticRegression(class_weight="balanced", max_iter=1000)
    model.fit(scaler.transform(X[train_mask]), y[train_mask])

    test = feat[test_mask].copy()
    test["score"] = model.predict_proba(scaler.transform(X[test_mask]))[:, 1]
    test["rank"] = test["score"].rank(ascending=False)

    print(f"\n=== {name}: train {train_mask.sum()} rows "
          f"({y[train_mask].sum()} attacks), test {test_mask.sum()} rows "
          f"({y[test_mask].sum()} attacks) ===")
    print("Coefficients (standardized):")
    print(dict(zip(cols, model.coef_[0].round(2))))
    print("Ranks of test attack windows:",
          sorted(test[test["label"] == 1]["rank"].astype(int).tolist()))

run(feat["window"] < cut, "train early, test late")
run(feat["window"] >= cut, "train late, test early")


=== train early, test late: train 4391 rows (6 attacks), test 3574 rows (6 attacks) ===
Coefficients (standardized):
{'flows': np.float64(-1.91), 'n_dst_ports': np.float64(-3.63), 'n_dst_ips': np.float64(-5.23), 'flows_per_dst_ip': np.float64(5.11), 'flows_per_dst_port': np.float64(-2.7), 'mean_fwd_pkts': np.float64(-1.13), 'mean_bwd_pkts': np.float64(1.46), 'dur_s': np.float64(3.42), 'no_reply_frac': np.float64(-4.84)}
Ranks of test attack windows: [1, 3, 4, 5, 7, 11]

=== train late, test early: train 3574 rows (6 attacks), test 4391 rows (6 attacks) ===
Coefficients (standardized):
{'flows': np.float64(-0.99), 'n_dst_ports': np.float64(-0.14), 'n_dst_ips': np.float64(-2.89), 'flows_per_dst_ip': np.float64(2.08), 'flows_per_dst_port': np.float64(-1.49), 'mean_fwd_pkts': np.float64(-1.49), 'mean_bwd_pkts': np.float64(-0.55), 'dur_s': np.float64(3.78), 'no_reply_frac': np.float64(-5.08)}
Ranks of test attack windows: [1, 18, 161, 217, 468, 473]


In [15]:
def run2(train_mask, cols, name, C=0.1):
    X2 = np.log1p(feat[cols].clip(lower=0))
    test_mask = ~train_mask
    scaler = StandardScaler().fit(X2[train_mask])
    model = LogisticRegression(class_weight="balanced", C=C, max_iter=1000)
    model.fit(scaler.transform(X2[train_mask]), y[train_mask])

    test = feat[test_mask].copy()
    test["score"] = model.predict_proba(scaler.transform(X2[test_mask]))[:, 1]
    test["rank"] = test["score"].rank(ascending=False).astype(int)

    print(f"\n=== {name} ===")
    print(dict(zip(cols, model.coef_[0].round(2))))
    print(test[test["label"] == 1].sort_values("rank")
          [["window", "flows", "n_dst_ports", "rank"]].to_string(index=False))

small = ["n_dst_ips", "n_dst_ports", "no_reply_frac", "mean_bwd_pkts"]
run2(feat["window"] < cut, small, "train early, test late")
run2(feat["window"] >= cut, small, "train late, test early")


=== train early, test late ===
{'n_dst_ips': np.float64(-3.75), 'n_dst_ports': np.float64(1.09), 'no_reply_frac': np.float64(-1.83), 'mean_bwd_pkts': np.float64(2.2)}
             window  flows  n_dst_ports  rank
2017-07-07 03:10:00   2011         1000     2
2017-07-07 03:05:00   9027          999     3
2017-07-07 03:20:00   9025          999     4
2017-07-07 02:55:00  45397          996     5
2017-07-07 02:50:00  93097          998     7
2017-07-07 03:00:00      2            1   972

=== train late, test early ===
{'n_dst_ips': np.float64(-1.32), 'n_dst_ports': np.float64(0.59), 'no_reply_frac': np.float64(-2.28), 'mean_bwd_pkts': np.float64(-0.18)}
             window  flows  n_dst_ports  rank
2017-07-07 01:55:00    177            5    87
2017-07-07 02:00:00     49            5    87
2017-07-07 01:50:00      1            1  1207
2017-07-07 02:10:00     70            5  1862
2017-07-07 02:15:00     68            5  1883
2017-07-07 01:05:00      7            1  1893


In [16]:
from sklearn.ensemble import IsolationForest

iso_cols = ["flows", "n_dst_ports", "n_dst_ips",
            "no_reply_frac", "mean_bwd_pkts", "dur_s"]
Xi = np.log1p(feat[iso_cols].clip(lower=0))

iso = IsolationForest(n_estimators=300, random_state=0).fit(Xi)
feat["iso_score"] = -iso.score_samples(Xi)    # higher = more anomalous
feat["iso_rank"] = feat["iso_score"].rank(ascending=False, method="first").astype(int)

print(feat[feat["label"] == 1].sort_values("iso_rank")
      [["window", "flows", "n_dst_ports", "iso_rank"]].to_string(index=False))
print("P@5:", precision_at_k(feat, "iso_score", 5),
      "R@12:", round(recall_at_k(feat, "iso_score", 12), 2))

# host-level view: each host gets its worst window
host = feat.groupby("Source IP")["iso_score"].max().sort_values(ascending=False)
print(host.head(10))
print("Rank of 172.16.0.1:", list(host.index).index("172.16.0.1") + 1, "of", len(host))

             window  flows  n_dst_ports  iso_rank
2017-07-07 02:50:00  93097          998         9
2017-07-07 03:10:00   2011         1000        21
2017-07-07 02:55:00  45397          996        31
2017-07-07 03:05:00   9027          999        40
2017-07-07 03:20:00   9025          999        41
2017-07-07 02:15:00     68            5       436
2017-07-07 01:05:00      7            1       437
2017-07-07 02:10:00     70            5       438
2017-07-07 01:50:00      1            1       554
2017-07-07 03:00:00      2            1       563
2017-07-07 01:55:00    177            5       831
2017-07-07 02:00:00     49            5      1005
P@5: 0.0 R@12: 0.08
Source IP
192.168.10.14    0.731287
192.168.10.17    0.727837
192.168.10.19    0.726019
192.168.10.50    0.723641
192.168.10.16    0.720970
172.16.0.1       0.717936
192.168.10.25    0.717647
192.168.10.9     0.716898
192.168.10.8     0.716659
192.168.10.5     0.713448
Name: iso_score, dtype: float64
Rank of 172.16.0.1: 6 of 366

In [17]:
# z-scores on the log scale: how far each feature is from the typical window
Z = (Xi - Xi.mean()) / Xi.std()
Z["Source IP"] = feat["Source IP"]
Z["window"] = feat["window"]

show = ["Source IP", "window"] + iso_cols
top = feat.sort_values("iso_score", ascending=False).head(6).index
atk = feat[feat["label"] == 1].sort_values("iso_score", ascending=False).head(3).index

print("Top 6 windows by Isolation Forest score:")
print(Z.loc[top, show].round(1).to_string(index=False))
print("\nThree highest-scoring attack windows:")
print(Z.loc[atk, show].round(1).to_string(index=False))

# host-level rank of the attacker under each method
for col in ["flows", "n_dst_ports", "iso_score"]:
    h = feat.groupby("Source IP")[col].max().sort_values(ascending=False)
    print(col, "-> rank of 172.16.0.1:", list(h.index).index("172.16.0.1") + 1)

Top 6 windows by Isolation Forest score:
    Source IP              window  flows  n_dst_ports  n_dst_ips  no_reply_frac  mean_bwd_pkts  dur_s
192.168.10.14 2017-07-07 01:30:00    6.2          1.4        8.3           -0.8            4.7    4.6
192.168.10.17 2017-07-07 01:25:00    5.2          2.0        6.9           -0.8            4.7    5.2
192.168.10.19 2017-07-07 01:35:00    6.2          1.7        7.5           -0.3            4.0    3.8
192.168.10.50 2017-07-07 02:50:00    4.9          7.7        3.3           -0.9            0.6    0.7
192.168.10.50 2017-07-07 02:55:00    4.5          7.6        3.0           -0.8            1.0    1.3
192.168.10.16 2017-07-07 03:25:00    5.3          1.9        7.1           -0.8            4.7    4.7

Three highest-scoring attack windows:
 Source IP              window  flows  n_dst_ports  n_dst_ips  no_reply_frac  mean_bwd_pkts  dur_s
172.16.0.1 2017-07-07 02:50:00    9.9          9.8        0.4           -1.0            0.4   -0.1
172.16.0

/tmp/ipykernel_200662/3699125790.py:11: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(Z.loc[top, show].round(1).to_string(index=False))
/tmp/ipykernel_200662/3699125790.py:13: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(Z.loc[atk, show].round(1).to_string(index=False))


In [18]:
Z = (Xi - Xi.mean()) / Xi.std()

best = feat.loc[feat.groupby("Source IP")["iso_score"].idxmax()]
best = best.sort_values("iso_score", ascending=False).head(20)

def why(i):
    z = Z.loc[i, iso_cols].abs().sort_values(ascending=False).head(2)
    return ", ".join(f"{c} ({Z.loc[i, c]:+.1f} sd)" for c in z.index)

report = pd.DataFrame({
    "rank": range(1, len(best) + 1),
    "host": best["Source IP"].values,
    "window": best["window"].dt.strftime("%H:%M").values,
    "score": best["iso_score"].round(3).values,
    "why": [why(i) for i in best.index],
    "is_attack": best["label"].values,   # for our checking only; Priya wouldn't have this
})
print(report.to_string(index=False))
print("Attackers in top 20:", report["is_attack"].sum())

 rank           host window  score                                      why  is_attack
    1  192.168.10.14  01:30  0.731     n_dst_ips (+8.3 sd), flows (+6.2 sd)          0
    2  192.168.10.17  01:25  0.728     n_dst_ips (+6.9 sd), dur_s (+5.2 sd)          0
    3  192.168.10.19  01:35  0.726     n_dst_ips (+7.5 sd), flows (+6.2 sd)          0
    4  192.168.10.50  02:50  0.724   n_dst_ports (+7.7 sd), flows (+4.9 sd)          0
    5  192.168.10.16  03:25  0.721     n_dst_ips (+7.1 sd), flows (+5.3 sd)          0
    6     172.16.0.1  02:50  0.718   flows (+9.9 sd), n_dst_ports (+9.8 sd)          1
    7  192.168.10.25  03:25  0.718 mean_bwd_pkts (+7.4 sd), dur_s (+4.5 sd)          0
    8   192.168.10.9  01:15  0.717     n_dst_ips (+6.6 sd), flows (+5.1 sd)          0
    9   192.168.10.8  02:40  0.717     n_dst_ips (+6.6 sd), flows (+5.5 sd)          0
   10   192.168.10.5  03:20  0.713     n_dst_ips (+6.1 sd), dur_s (+5.2 sd)          0
   11  192.168.10.15  02:25  0.713     n_ds

In [19]:
w = df[(df["Source IP"] == "192.168.10.50") &
       (df["window"] == pd.Timestamp("2017-07-07 02:50:00"))]

print("Flows from 192.168.10.50 in that window:", len(w))
print(w["Destination IP"].value_counts().head(5))
print("Dest ports (top 5):")
print(w["Destination Port"].value_counts().head(5))

# who was talking TO 192.168.10.50 in that window?
inbound = df[(df["Destination IP"] == "192.168.10.50") &
             (df["window"] == pd.Timestamp("2017-07-07 02:50:00"))]
print("\nInbound flows:", len(inbound))
print(inbound["Source IP"].value_counts().head(5))
print(inbound["Label"].value_counts())

Flows from 192.168.10.50 in that window: 560
Destination IP
172.16.0.1       488
192.168.10.3      20
192.168.10.17      9
192.168.10.19      7
192.168.10.14      6
Name: count, dtype: int64
Dest ports (top 5):
Destination Port
53       16
42668    11
33830    11
63910    10
41653     9
Name: count, dtype: int64

Inbound flows: 93169
Source IP
172.16.0.1       93096
192.168.10.17       12
192.168.10.14       10
192.168.10.15        9
192.168.10.16        9
Name: count, dtype: int64
Label
PortScan    93096
BENIGN         73
Name: count, dtype: int64
